
# Marine Drifter Prediction with Transformer + Neural ODE

This notebook implements a Physics-Residual Transformer-Neural ODE model for marine drifter trajectory prediction.

Core idea:

\[
\Delta x_{pred}
=
\Delta x_{adv}
+
r_\theta
\]

Where:

- Advection baseline models first-order physical motion
- Transformer extracts temporal dynamics
- Neural ODE models continuous latent evolution
- Residual head learns nonlinear corrections


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Imports

In [19]:
!pip install torchdiffeq

import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_squared_error

from torchdiffeq import odeint

from tqdm import tqdm


## Load Dataset

In [ ]:

def load_year_npz(npz_path):

    data = np.load(npz_path, allow_pickle=True)

    return {
        "X_train": data["X_train"],
        "y_train": data["y_train"],

        "X_val": data["X_val"],
        "y_val": data["y_val"],

        "X_test": data["X_test"],
        "y_test": data["y_test"],

        "feature_cols": data["feature_cols"]
    }

base_dir = "/content/drive/MyDrive/2022-2025_processed_data_extended"

years = [2022, 2023, 2024, 2025]

train_X_list = []
train_y_list = []

val_X_list = []
val_y_list = []

test_X_list = []
test_y_list = []

for year in years:

    npz_path = os.path.join(
        base_dir,
        f"{year}_extended",
        f"drifter_{year}_extended_supervised_windows.npz"
    )

    d = load_year_npz(npz_path)

    train_X_list.append(d["X_train"])
    train_y_list.append(d["y_train"])

    val_X_list.append(d["X_val"])
    val_y_list.append(d["y_val"])

    test_X_list.append(d["X_test"])
    test_y_list.append(d["y_test"])

    feature_cols = d["feature_cols"]

X_train = np.concatenate(train_X_list, axis=0)
y_train = np.concatenate(train_y_list, axis=0)

X_val = np.concatenate(val_X_list, axis=0)
y_val = np.concatenate(val_y_list, axis=0)

X_test = np.concatenate(test_X_list, axis=0)
y_test = np.concatenate(test_y_list, axis=0)

print(X_train.shape)
print(y_train.shape)


## Feature Selection

In [ ]:

selected_features = [
    "latitude",
    "longitude",
    "ve",
    "vn",
    "speed",
    "direction",
    "accel_east",
    "accel_north"
]

feature_cols = list(feature_cols)

feature_idx = [
    feature_cols.index(f)
    for f in selected_features
]

X_train = X_train[:, :, feature_idx]
X_val = X_val[:, :, feature_idx]
X_test = X_test[:, :, feature_idx]

print(X_train.shape)


## Advection Baseline

In [ ]:

DT_SECONDS = 24 * 3600
METERS_PER_DEG_LAT = 111000

VE_IDX = selected_features.index("ve")
VN_IDX = selected_features.index("vn")

def compute_advection_baseline(X):

    ve = X[:, -1, VE_IDX]
    vn = X[:, -1, VN_IDX]

    delta_lat = (vn * DT_SECONDS) / METERS_PER_DEG_LAT
    delta_lon = (ve * DT_SECONDS) / METERS_PER_DEG_LAT

    baseline = np.stack(
        [delta_lat, delta_lon],
        axis=1
    )

    return baseline

adv_train = compute_advection_baseline(X_train)
adv_val = compute_advection_baseline(X_val)
adv_test = compute_advection_baseline(X_test)

residual_train = y_train - adv_train
residual_val = y_val - adv_val
residual_test = y_test - adv_test


## Dataset

In [ ]:

class DrifterDataset(Dataset):

    def __init__(self, X, target_y, adv):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(target_y, dtype=torch.float32)
        self.adv = torch.tensor(adv, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx],
            self.adv[idx]
        )


## Positional Encoding

In [ ]:

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=32):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]


## Transformer Encoder

In [25]:

class TransformerEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=128,
        nhead=4,
        num_layers=2
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            input_dim,
            d_model
        )

        self.pe = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        x = self.input_proj(x)
        x = self.pe(x)

        h = self.encoder(x)

        return h[:, -1]


## Neural ODE

In [26]:

class ODEFunc(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, t, z):

        return self.net(z)


class ODEBlock(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.func = ODEFunc(hidden_dim)

    def forward(self, z0):

        t = torch.tensor(
            [0, 1],
            dtype=torch.float32,
            device=z0.device
        )

        z_t = odeint(
            self.func,
            z0,
            t,
            method='rk4'
        )

        return z_t[-1]


## Full Model

In [27]:

class TransformerODEModel(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.encoder = TransformerEncoder(
            input_dim=input_dim
        )

        self.ode = ODEBlock(128)

        self.head = nn.Sequential(

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x, adv):

        z = self.encoder(x)

        z = self.ode(z)

        residual = self.head(z)

        pred = adv + residual

        return pred, residual


## Loss Functions

In [28]:

criterion = nn.MSELoss()

def physics_loss(residual):

    return torch.mean(residual ** 2)

def total_loss(
    pred,
    target,
    residual,
    lambda_phys=0.01
):

    mse = criterion(pred, target)

    phys = physics_loss(residual)

    return mse + lambda_phys * phys


## DataLoaders

In [29]:

train_dataset = DrifterDataset(
    X_train,
    y_train,
    adv_train
)

val_dataset = DrifterDataset(
    X_val,
    y_val,
    adv_val
)

test_dataset = DrifterDataset(
    X_test,
    y_test,
    adv_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256
)


## Training Setup

In [30]:

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = TransformerODEModel(
    input_dim=len(selected_features)
).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

print(device)


cuda


## Training Loop

In [38]:
EPOCHS = 150
save_every = 10

## Resume Training from Checkpoint

To allow for continuous training and prevent loss of progress, the training loop will now check for existing checkpoints and resume from the latest one if found. This involves:

1.  **Finding the latest checkpoint**: The code will scan the `checkpoints` directory for files named `model_X.pt` and identify the one with the highest epoch number.
2.  **Loading states**: If a checkpoint is found, the `model`'s state dictionary and the `optimizer`'s state dictionary will be loaded.
3.  **Adjusting starting epoch**: The training loop will then begin from the epoch immediately following the one saved in the checkpoint.

In [39]:
import os

os.makedirs("checkpoints", exist_ok=True)

##################################################
# Improved Advection Baseline
##################################################

DT_SECONDS = 24 * 3600
METERS_PER_DEG_LAT = 111000

VE_IDX = selected_features.index("ve")
VN_IDX = selected_features.index("vn")
LAT_IDX = selected_features.index("latitude")

def compute_advection_baseline(X):

    ve = X[:, -1, VE_IDX]
    vn = X[:, -1, VN_IDX]

    lat = X[:, -1, LAT_IDX]

    lat_rad = np.deg2rad(lat)

    delta_lat = (
        vn * DT_SECONDS
    ) / METERS_PER_DEG_LAT

    delta_lon = (
        ve * DT_SECONDS
    ) / (
        METERS_PER_DEG_LAT *
        np.cos(lat_rad)
    )

    baseline = np.stack(
        [delta_lat, delta_lon],
        axis=1
    )

    return baseline


##################################################
# Recompute Advection Targets
##################################################

adv_train = compute_advection_baseline(X_train)
adv_val = compute_advection_baseline(X_val)
adv_test = compute_advection_baseline(X_test)


##################################################
# Validation Function
##################################################

def evaluate(model, loader):

    model.eval()

    total_loss_eval = 0

    with torch.no_grad():

        for Xb, yb, advb in loader:

            Xb = Xb.to(device)
            yb = yb.to(device)
            advb = advb.to(device)

            pred, residual = model(Xb, advb)

            loss = total_loss(
                pred,
                yb,
                residual
            )

            total_loss_eval += loss.item()

    return total_loss_eval / len(loader)


##################################################
# Optimizer
##################################################

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)


##################################################
# Scheduler
##################################################

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)


In [40]:
def find_latest_checkpoint(checkpoint_dir="checkpoints"):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("model_") and f.endswith(".pt")]
    if not checkpoints:
        return None

    # Sort by epoch number to get the latest
    checkpoints.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))
    return os.path.join(checkpoint_dir, checkpoints[-1])

latest_checkpoint_path = find_latest_checkpoint()
start_epoch_idx = 0 # 0-indexed epoch to start training from

if latest_checkpoint_path:
    print(f"Resuming training from checkpoint: {latest_checkpoint_path}")
    checkpoint = torch.load(latest_checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    # checkpoint["epoch"] is 1-indexed, so if it's N, it means epoch N was COMPLETED.
    # We should start training from epoch N+1, which is 0-indexed N.
    start_epoch_idx = checkpoint["epoch"]
    print(f"Starting training from epoch {start_epoch_idx + 1}")


# Adjust the training loop to use the `start_epoch_idx`
# The original loop was `for epoch in range(EPOCHS):` where epoch was 0-indexed.
# Now, `current_epoch_idx` will be 0-indexed.
for current_epoch_idx in range(start_epoch_idx, EPOCHS):

    ##############################################
    # Train
    ##############################################

    model.train()

    train_loss = 0

    for Xb, yb, advb in tqdm(train_loader):

        Xb = Xb.to(device)
        yb = yb.to(device)
        advb = advb.to(device)

        optimizer.zero_grad()

        pred, residual = model(Xb, advb)

        loss = total_loss(
            pred,
            yb,
            residual
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    ##############################################
    # Validation
    ##############################################

    val_loss = evaluate(
        model,
        val_loader
    )

    ##############################################
    # Scheduler Step
    ##############################################

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']

    ##############################################
    # Print
    ##############################################

    print(
        f"Epoch {current_epoch_idx+1} | " # Print 1-indexed epoch number
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"LR: {current_lr:.2e}"
    )

    ##############################################
    # Save Latest
    ##############################################
    if (current_epoch_idx + 1) % save_every == 0:
      torch.save({

          "epoch": current_epoch_idx + 1, # Save as 1-indexed epoch number

          "model_state_dict":
              model.state_dict(),

          "optimizer_state_dict":
              optimizer.state_dict(),

          "train_loss":
              train_loss,

          "val_loss":
              val_loss

      }, f"checkpoints/model_{current_epoch_idx + 1}.pt")


Resuming training from checkpoint: checkpoints/model_50.pt
Starting training from epoch 51


100%|██████████| 2843/2843 [00:40<00:00, 70.02it/s]


Epoch 51 | Train Loss: 0.008316 | Val Loss: 0.008741 | LR: 1.56e-06


100%|██████████| 2843/2843 [00:40<00:00, 70.55it/s]


Epoch 52 | Train Loss: 0.008315 | Val Loss: 0.008743 | LR: 1.56e-06


...

Epoch 83 | Train Loss: 0.008264 | Val Loss: 0.008679 | LR: 2.44e-08


100%|██████████| 2843/2843 [00:39<00:00, 72.08it/s]


Epoch 84 | Train Loss: 0.008278 | Val Loss: 0.008681 | LR: 2.44e-08


 52%|█████▏    | 1479/2843 [00:20<00:19, 71.75it/s]


KeyboardInterrupt: 

## Evaluation

In [41]:
model.eval()

# Find and load the latest checkpoint for evaluation
latest_checkpoint_path = find_latest_checkpoint()
if latest_checkpoint_path:
    print(f"Loading model from checkpoint for evaluation: {latest_checkpoint_path}")
    checkpoint = torch.load(latest_checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    print("No checkpoint found. Evaluating with the current model state.")

preds = []
targets = []

with torch.no_grad():

    for Xb, yb, advb in test_loader:

        Xb = Xb.to(device)
        advb = advb.to(device)

        pred, _ = model(Xb, advb)

        preds.append(pred.cpu().numpy())
        targets.append(yb.numpy())

preds = np.concatenate(preds)
targets = np.concatenate(targets)

rmse_lat = np.sqrt(
    mean_squared_error(
        targets[:, 0],
        preds[:, 0]
    )
)

rmse_lon = np.sqrt(
    mean_squared_error(
        targets[:, 1],
        preds[:, 1]
    )
)

print("RMSE Latitude:", rmse_lat)
print("RMSE Longitude:", rmse_lon)


Loading model from checkpoint for evaluation: checkpoints/model_80.pt
RMSE Latitude: 0.0595855615203107
RMSE Longitude: 0.7358046879562262
